<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/7.%20Advanced%20AI%20Agents/12.%20Dataset%20Engineering%20%26%26%20Inference%20Optimization%20/Live_Class_Notebook_Inference_July_23_2026.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

https://www.youtube.com/watch?v=Tsvxx-GGlTg

https://www.csfieldguide.org.nz/en/interactives/rgb-mixer/

https://www.csfieldguide.org.nz/en/interactives/pixel-viewer/

https://ezyang.github.io/convolution-visualizer/

In [1]:
import os
import gc
import time
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

print(f"device: {device}")
print(f"dtype:  {dtype}")
if device != "cuda":
    print("⚠️  Running on CPU. Memory measurements in the KV cache section won't work; "
          "consider switching to a GPU runtime (Runtime → Change runtime type → T4).")

device: cuda
dtype:  torch.bfloat16


In [2]:
MODEL_NAME = "gpt2"  # 124M params — fits anywhere

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
model.config.pad_token_id = model.config.eos_token_id
model.eval()
print("Loaded.")

Loading gpt2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded.


In [5]:
prompt = "Donald Trump is complete"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
print(f'Prompt: "{prompt}"')

Prompt: "Donald Trump is complete"


In [6]:
print("Running model.generate() with use_cache=True ...")

start = time.perf_counter()
with torch.no_grad():
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=False,
    )
elapsed = time.perf_counter() - start

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("--- Generated ---")
print(generated_text)
print(f"\n({elapsed:.3f}s for 50 tokens)")

Running model.generate() with use_cache=True ...
--- Generated ---
Donald Trump is complete and utter garbage.

The president-elect is a complete and utter garbage.

The president-elect is a complete and utter garbage.

The president-elect is a complete and utter garbage.

The president-elect is

(1.286s for 50 tokens)
